# Logarithmic Transformations & Quadratic Terms: Exercises
### Applied Statistical Data Analysis. Prof. Dr. Kristyna Ters | MSc Finance | FHNW

---
> **Instructions:**
> - Work through the exercises in order; each builds on the previous one
> - Fill in your code in the cells marked with `# YOUR CODE HERE`
> - Answer written questions by double-clicking the markdown cell and editing it
> - Run cells with **Shift+Enter**
> - Solutions will be released after the submission deadline

In [ ]:
!pip install yfinance statsmodels --quiet

import yfinance as yf
import pandas as pd
import numpy as np
import statsmodels.api as sm
from statsmodels.stats.diagnostic import linear_reset
import matplotlib.pyplot as plt
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.facecolor':'white', 'axes.facecolor':'white',
    'axes.spines.top':False, 'axes.spines.right':False,
    'axes.grid':True, 'grid.alpha':0.3, 'font.size':11
})
YELLOW = '#FDE70E'; ORANGE = '#FCB310'; RED = '#C70101'
GREY   = '#4B4B4B'; BLUE = '#0E75FE'; GREEN = '#0B7A3C'
print('✓ Libraries loaded.')

---
# Exercise 1: Name the Form (Warm-Up, No Code)

For each estimated equation, name the functional form (lin-lin, log-lin, lin-log, log-log) and interpret the highlighted coefficient in ONE precise sentence.

| # | Estimated equation | Interpret |
|---|--------------------|-----------|
| a | returns = 0.02 + **1.15**·mkt_returns | 1.15 |
| b | ln(price) = 4.60 **− 0.096**·yield | −0.096 |
| c | ln(volume) = 0.9 + **1.08**·ln(mcap) | 1.08 |
| d | salary = 20 + **0.45**·ln(AUM) | 0.45 (salary in kCHF) |
| e | ln(wage) = ... + **0.28**·D_CFA | 0.28 |

**Your answers** (double-click to edit):

| # | Form | One-sentence interpretation |
|---|------|-----------------------------|
| a | | |
| b | | |
| c | | |
| d | | |
| e | | |

---
# Data for Exercises 2 to 6: the Swiss Cross-Section

Run this provided cell once (it is the same loader as in the Lecture Notebook).

In [ ]:
# Provided: build the Swiss cross-section (same code as the Lecture Notebook)
SMI = ['NESN.SW','ROG.SW','NOVN.SW','UBSG.SW','ZURN.SW','CFR.SW','ABBN.SW','SIKA.SW',
       'LONN.SW','ALC.SW','GIVN.SW','HOLN.SW','SLHN.SW','PGHN.SW','SCMN.SW','SREN.SW',
       'GEBN.SW','SOON.SW','LOGN.SW','KNIN.SW']
SMIM = ['BAER.SW','ADEN.SW','CLN.SW','TEMN.SW','VACN.SW','STMN.SW','SCHP.SW','GALE.SW',
        'HELN.SW','PSPN.SW','ALLN.SW','BARN.SW','EMSN.SW','SGSN.SW','LISP.SW','DKSH.SW',
        'SFSN.SW','BUCN.SW','SUN.SW','AVOL.SW']

rows, dropped = [], []
for tick in SMI + SMIM:
    try:
        t  = yf.Ticker(tick)
        px = t.history(period='6mo')
        if len(px) < 60:
            dropped.append((tick, 'short history')); continue
        vol_chf = float((px['Close'] * px['Volume']).mean())
        mcap    = t.fast_info['marketCap']
        if not mcap or mcap <= 0 or vol_chf <= 0:
            dropped.append((tick, 'no market cap / no volume')); continue
        rows.append({'ticker': tick, 'mcap': mcap, 'volume': vol_chf,
                     'D_SMI': 1.0 if tick in SMI else 0.0})
    except Exception as e:              # never swallow silently: n drives every result below
        dropped.append((tick, type(e).__name__))

if dropped:
    print(f'{len(dropped)} tickers dropped: {dropped}')
if not rows:
    raise RuntimeError('No tickers downloaded. Yahoo throttles ~40 sequential Ticker '
                       'calls - wait a minute and re-run, or shorten the list.')

df = pd.DataFrame(rows).set_index('ticker')
df['ln_vol']  = np.log(df['volume'])
df['ln_mcap'] = np.log(df['mcap'])
print(f'n = {len(df)} stocks ({int(df.D_SMI.sum())} SMI)')

---
# Exercise 2: The Elasticity

Run the provided loader cell, then estimate the log-log model WITHOUT the dummy:

$$\ln V_i = \beta_0 + \beta_1 \ln M_i + u_i$$

Report the elasticity with a 95% confidence interval and interpret it in one sentence.

**Written question:** your neighbour estimated the same model on US stocks in dollars. Why are your two elasticities directly comparable even though the currencies differ?

In [ ]:
# YOUR CODE HERE
# X = sm.add_constant(df['ln_mcap']); OLS with HC1
# print beta_1, its 95% CI (m.conf_int()), one-sentence interpretation


---
# Exercise 3: The Dummy, Naive vs Exact

Add the SMI dummy and compute BOTH readings of its coefficient: the naive percentage (100·δ) and the exact percentage effect (100·(e^δ − 1)).

**Written question:** for which magnitudes of δ is the naive reading acceptable, and why exactly does it fail for large δ?

In [ ]:
# YOUR CODE HERE
# X = sm.add_constant(df[['ln_mcap', 'D_SMI']]); OLS with HC1
# delta = params['D_SMI']; naive = 100*delta; exact = 100*(np.exp(delta)-1)


---
# Exercise 4: Swap the Reference Category

Re-estimate the model with a MID-CAP dummy instead (D_MID = 1 − D_SMI).

(a) What is the new dummy coefficient, and how does it relate to the old one?
(b) Compute the exact percentage effect of being a mid cap. Explain why it is NOT simply minus the SMI effect.
(c) Verify numerically: (1 + g_up)·(1 + g_down) = 1, where g are the two exact effects as decimals.

In [ ]:
# YOUR CODE HERE
# df['D_MID'] = 1 - df['D_SMI']; re-estimate; compare coefficients and exact effects


---
# Exercise 5: RESET as Referee

Estimate the LEVELS specification (volume on mcap and the dummy, no logs) and run RESET on both the levels model and your log-log model from Exercise 3.

**Written question:** why would comparing the two R² values NOT be a valid way to choose between these models?

In [ ]:
# YOUR CODE HERE
# m_lvl = OLS(volume on [mcap, D_SMI]); linear_reset(..., power=3, use_f=True) on both models


---
# Exercise 6: A lin-log Variant

Regress volume IN MILLIONS (levels) on ln(mcap): a lin-log model. Interpret the slope in one sentence, being precise about units.

**Written question:** when might a lin-log form be economically more natural than log-log?

In [ ]:
# YOUR CODE HERE
# y = df['volume']/1e6; X = sm.add_constant(df['ln_mcap']); interpret slope/100


---
# Exercise 7: The Quadratic Fund Model

Simulate the fund sample with the SAME seed as the lecture (so your numbers match), then estimate alpha on size and size squared with HC1 standard errors. Report both coefficients with t-statistics and state the shape.

```python
rng = np.random.default_rng(42)
funds = pd.DataFrame({'size': rng.uniform(0.05, 2.2, 120)})
funds['alpha'] = -0.2 + 2.4*funds['size'] - 1.5*funds['size']**2 + rng.normal(0, 0.55, 120)
```

In [ ]:
# YOUR CODE HERE
# build funds as above; funds['size2'] = size**2; OLS with HC1; report and interpret signs


---
# Exercise 8: Turning Point and Marginal Effects

(a) Compute the turning point x* = −β₁/(2β₂) and check whether it lies inside the data range.
(b) Compute the marginal effect β₁ + 2β₂x at sizes 0.2, at x*, and at 1.5 bn.
(c) Write ONE sentence a fund allocator could quote.

In [ ]:
# YOUR CODE HERE


---
# Exercise 9: What the Linear Model Would Have Told You

Fit the WRONG model, alpha on size only (no square), and run RESET on it.

**Written question:** what conclusion about fund size would the linear model suggest, why is it misleading, and how does RESET warn you?

In [ ]:
# YOUR CODE HERE
# m_lin = OLS(alpha on size); slope? RESET?


---
# Exercise 10: The Reporting Memo (Written)

In at most six sentences, write the results memo for both models: the elasticity and the SMI effect (with the exact percentage), the fund-size curve (marginal effects and turning point), and one sentence on why you chose these functional forms. Imagine the reader is a portfolio manager, not an econometrician.

---
*Applied Statistical Data Analysis | Prof. Dr. Kristyna Ters | FHNW School of Business | HS 2026*